# IO Cloud Agent Cloud — MCP 教学案例

## 什么是 Agent Cloud？

[io.net](https://io.net) 提供了一个 **MCP (Model Context Protocol) 服务器**，让 AI 代理（Claude Code、Cursor、Windsurf 等）可以通过自然语言直接管理去中心化 GPU 基础设施。

**本教程你将学到：**

| 步骤 | 功能 | 是否花钱 |
|------|------|----------|
| 1 | 连接 MCP 服务器，列出所有可用工具 | 免费 |
| 2 | 浏览 CaaS 硬件目录 | 免费 |
| 3 | 估算部署价格 | 免费 |
| 4 | **部署一个最便宜的容器** | **花钱** |
| 5 | 查看部署状态 & 容器详情 | 免费 |
| 6 | 列出所有部署 | 免费 |
| 7 | **销毁部署（省钱！）** | 免费 |

> 注意：步骤 4 会产生实际费用，我们会选最便宜的配置（1 GPU、1 副本、1 小时），并在最后及时销毁。

## 0. 环境准备

In [ ]:
import os
# 如果需要代理才能访问外网，取消下面两行的注释
# os.environ['http_proxy']  = 'http://127.0.0.1:7890'
# os.environ['https_proxy'] = 'http://127.0.0.1:7890'

In [ ]:
import asyncio, json, time
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# ============================================================
#  填入你的 io.net API Key（需要 io-cloud project 权限）
# ============================================================
API_KEY = 'io-v2-***'

MCP_URL = 'https://mcp.io.solutions/mcp'
HEADERS = {'x-api-key': API_KEY}

print('OK')

### 辅助函数

In [ ]:
async def call_tool(tool_name, arguments=None, retries=3):
    """连接 MCP 服务器并调用指定工具，带重试。"""
    for attempt in range(retries):
        try:
            async with streamablehttp_client(MCP_URL, headers=HEADERS, timeout=60) as (r, w, _):
                async with ClientSession(r, w) as session:
                    await session.initialize()
                    result = await session.call_tool(tool_name, arguments=arguments or {})
                    text = result.content[0].text if result.content else ''
                    try:
                        return json.loads(text)
                    except json.JSONDecodeError:
                        return text
        except Exception as e:
            if attempt < retries - 1:
                print(f'  重试 {attempt+1}/{retries}: {type(e).__name__}')
                await asyncio.sleep(2)
            else:
                raise


async def list_all_tools():
    async with streamablehttp_client(MCP_URL, headers=HEADERS, timeout=60) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            return tools.tools


def pretty(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))


def extract_data(resp):
    """从嵌套的 data.data 结构中提取实际数据。"""
    if isinstance(resp, dict) and 'data' in resp:
        inner = resp['data']
        if isinstance(inner, dict) and 'data' in inner:
            return inner['data']
        return inner
    return resp


print('OK')

---

## 1. 连接 MCP 服务器 - 列出所有可用工具 (免费)

先确认能连上，同时了解 IO Cloud 提供了哪些能力。

In [ ]:
tools = await list_all_tools()

print(f'共发现 {len(tools)} 个工具:')
print()
for t in tools:
    prefix = '[CaaS]' if t.name.startswith('caas') else '[VMaaS]'
    print(f'  {prefix}  {t.name}')
    print(f'          {t.description}')
    print()

---

## 2. 浏览 CaaS 硬件目录 (免费)

使用 `caas_get_hardware_ids` 查看当前可用的 GPU 型号、价格、库存。

In [ ]:
caas_hw = await call_tool('caas_get_hardware_ids')
items = extract_data(caas_hw)

# items 可能是 list 或 dict，统一处理
if isinstance(items, dict):
    items = list(items.values()) if not any(isinstance(v, list) for v in items.values()) else next(v for v in items.values() if isinstance(v, list))

print(f'共 {len(items)} 种硬件配置')
print()

# 按价格排序
sorted_items = sorted(items, key=lambda x: x.get('price', 999) if isinstance(x, dict) else 999)

print(f'{"GPU":<25} {"hw_id":<8} {"$/hr":<10} {"avail":<8} {"location":<8}')
print('-' * 65)
for it in sorted_items[:15]:
    if isinstance(it, dict):
        print(f'{str(it.get("hardware_name","?")):<25} '
              f'{str(it.get("hardware_id","?")):<8} '
              f'${it.get("price",0):<9.2f} '
              f'{str(it.get("available","?")):<8} '
              f'{str(it.get("location","-")):<8}')

---

## 3. 估算部署价格 (免费)

在真正花钱之前，用 `caas_get_price_estimate` 先算一下要花多少。

我们选择 RTX 4090 (hw_id=12)，在美国 (location_id=2)，1 GPU、1 副本、1 小时。

In [ ]:
# RTX 4090: hardware_id=12, 美国: location_id=2
# 这些是经过验证的可用参数
HW_ID = 12       # GeForce RTX 4090, ~$0.30/hr
LOCATION_ID = 2  # United States

price_est = await call_tool('caas_get_price_estimate', {
    'location_ids': [LOCATION_ID],
    'hardware_id': HW_ID,
    'duration_hours': 1,
    'gpus_per_container': 1,
    'replica_count': 1
})

print('价格估算：')
pretty(price_est)

---

## 4. 部署一个容器（会花钱！）

现在我们来真正部署！配置：
- **硬件**：RTX 4090 (hw_id=12)
- **镜像**：`nginx:latest`
- **规格**：1 GPU、1 副本、1 小时

> **运行此 Cell 会产生真实费用**。完成后请务必执行后面的“销毁部署”步骤！

In [ ]:
deployment_name = f'tutorial-demo-{int(time.time()) % 100000}'

deploy_result = await call_tool('caas_deploy_container', {
    'request': {
        'resource_private_name': deployment_name,
        'duration_hours': 1,
        'gpus_per_container': 1,
        'hardware_id': HW_ID,
        'replica_count': 1,
        'traffic_port': 80,
        'image_url': 'nginx:latest',
        'location_ids': [LOCATION_ID]
    }
})

print(f'部署请求已发送！名称: {deployment_name}')
print()
pretty(deploy_result)

# 提取 deployment_id
deployment_id = None
dep_data = extract_data(deploy_result)
if isinstance(dep_data, dict):
    deployment_id = dep_data.get('id') or dep_data.get('deployment_id')
elif isinstance(deploy_result, dict):
    deployment_id = deploy_result.get('id') or deploy_result.get('deployment_id')

print(f'\nDeployment ID: {deployment_id}')
print('（请记住此 ID，后面查看状态和销毁都要用）')

---

## 5. 查看部署状态 & 容器详情 (免费)

部署提交后，容器需要一些时间启动。我们来检查状态。

In [ ]:
if deployment_id:
    status = await call_tool('caas_get_deployment', {
        'deployment_id': str(deployment_id)
    })
    print(f'部署 {deployment_id} 的状态：')
    print()
    pretty(status)
else:
    print('没有找到 deployment_id，请检查上一步的部署结果')
    print('你可以手动设置: deployment_id = "你的ID"')

In [ ]:
# 查看容器/Worker 详情
if deployment_id:
    containers = await call_tool('caas_get_deployment_containers', {
        'deployment_id': str(deployment_id)
    })
    print(f'部署 {deployment_id} 的容器列表：')
    print()
    pretty(containers)

---

## 6. 列出所有部署 (免费)

查看你账号下的所有 CaaS 部署。

In [ ]:
all_deployments = await call_tool('caas_list_deployments', {
    'page': 1,
    'page_size': 5
})

print('我的所有 CaaS 部署：')
print()
pretty(all_deployments)

---

## 7. 销毁部署（省钱！）

> **重要**：教程完成后务必销毁部署，否则会持续计费直到 `duration_hours` 到期！

In [ ]:
if deployment_id:
    destroy_result = await call_tool('caas_destroy_deployment', {
        'deployment_id': str(deployment_id)
    })
    print(f'销毁部署 {deployment_id}：')
    print()
    pretty(destroy_result)
else:
    print('没有 deployment_id')
    print('如果你知道 ID，可以手动执行：')
    print('deployment_id = "你的ID"  # 填入后重新运行此 Cell')

In [ ]:
# 确认已销毁
if deployment_id:
    final_status = await call_tool('caas_get_deployment', {
        'deployment_id': str(deployment_id)
    })
    print('销毁后的状态：')
    print()
    pretty(final_status)

---

## 总结

本教程演示了 IO Cloud MCP 服务器的核心功能：

| 工具 | 用途 |
|------|------|
| `caas_get_hardware_ids` | 查看 CaaS 支持的硬件类型和价格 |
| `caas_get_price_estimate` | 部署前估算价格 |
| `caas_deploy_container` | 部署容器集群 |
| `caas_get_deployment` | 查看单个部署的详细状态 |
| `caas_get_deployment_containers` | 查看部署内的容器/Worker |
| `caas_list_deployments` | 列出所有部署 |
| `caas_destroy_deployment` | 销毁部署，停止计费 |

### 在 AI 代理中使用

以上所有操作，都可以通过自然语言让 AI 代理完成，例如：

```
"帮我找最便宜的 4 卡 H100 集群并部署我的 PyTorch 镜像"
"列出我当前所有运行中的容器"
"把 tutorial-demo 那个部署销毁掉"
```